# STACK CATHEDRAL — Auto-Balancer
Reads gameplay logs, analyzes balance, outputs `balance_patch.json`.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

try:
    df = pd.read_csv('/kaggle/input/stack-cathedral-logs/gameplay_logs.csv')
    print(f'Loaded {len(df)} rows')
except:
    print('No logs found — using defaults')
    df = pd.DataFrame()

patch = {
    'enemy_hp_mult': {'drift': 1.0, 'chain': 1.0, 'conflict': 1.0, 'paren': 1.0, 'bracket': 1.0, 'brace': 1.0},
    'weapon_damage': {'apply': 26, 'destroy': 90, 'plan': 8, 'rollback': 999},
    'spawn': {'drones': 25, 'chains': 6, 'conflicts': 2, 'paren_pairs': 8, 'bracket_pairs': 6, 'brace_pairs': 5},
    'timing': {'wave_spacing': 25, 'boss_spawn_wave': 2, 'boss_frame_interval': 9},
    'player': {'hp': 100, 'shield_regen': 8, 'energy_regen': 7, 'speed': 45}
}

In [ ]:
if len(df) > 5:
    # WEAPON BALANCE
    for col, name in [('apply_kills','apply'),('destroy_kills','destroy'),('plan_kills','plan'),('rollback_kills','rollback')]:
        if col in df.columns:
            avg = df[col].mean()
            print(f'  {name}: {avg:.1f} avg kills/run')
            if avg < 2 and name != 'rollback':
                patch['weapon_damage'][name] = int(patch['weapon_damage'][name] * 1.20)
                print(f'    BUFFED → {patch["weapon_damage"][name]}')

    # DEATH RATE → difficulty adjust
    if 'deaths' in df.columns:
        avg_deaths = df['deaths'].mean()
        print(f'  Avg deaths: {avg_deaths:.1f}')
        if avg_deaths > 3:
            for k in patch['enemy_hp_mult']: patch['enemy_hp_mult'][k] = round(patch['enemy_hp_mult'][k] * 0.85, 2)
            print('    NERFING enemies')
        elif avg_deaths < 0.5:
            for k in patch['enemy_hp_mult']: patch['enemy_hp_mult'][k] = round(patch['enemy_hp_mult'][k] * 1.10, 2)
            print('    BUFFING enemies')

    # BRACKET CLOSE RATE
    if 'validate_closed' in df.columns and 'bracket_pairs_closed' in df.columns:
        rate = df['validate_closed'].sum() / max(1, df['bracket_pairs_total'].sum() if 'bracket_pairs_total' in df.columns else 19)
        print(f'  Close rate: {rate:.1%}')
        if rate < 0.25:
            patch['timing']['boss_frame_interval'] += 2
            print('    Slowing boss growth')

    # SURVIVAL TIME
    if 'time_seconds' in df.columns:
        avg_t = df['time_seconds'].mean()
        print(f'  Avg survival: {avg_t:.0f}s')
        if avg_t > 120:
            patch['timing']['wave_spacing'] = max(15, patch['timing']['wave_spacing'] - 5)
            patch['spawn']['drones'] += 3
            print('    FASTER + more drones')

Path('/kaggle/working/balance_patch.json').write_text(json.dumps(patch, indent=2))
print('\n=== BALANCE PATCH ===')
print(json.dumps(patch, indent=2))